# QA retrieval: tài liệu cha và Notes trước reranker

Cùng một Knowledge Base và cùng mã nguồn `rag/`. Notebook này so sánh **đường cũ** (fusion → reranker → bổ sung graph vào gói căn cứ) với **đường thử nghiệm** (fusion → tài liệu cha/Notes thật → reranker). Không chia câu hỏi bằng mẫu `if/else` theo tầng HS. Nhánh truy vấn bỏ mã nghi vấn chỉ chạy khi câu hỏi thực sự chứa mã bị phủ định hoặc đặt nghi vấn.

Chọn chính sách bằng **320 train**. Chỉ sau khi ghi cấu hình đã chọn mới mở **150 test**; 50 câu cũ được báo riêng như tập chẩn đoán, 100 câu mới là tập xác nhận độc lập. Output so sánh nằm trong notebook, không xuất report rời.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in candidates if (p / "rag/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import time

import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import config, digest, load_corpus, read, write_json
from rag.db import connect
from rag.embedding import Embedder
from rag.evaluation import (
    CONFIGURATIONS,
    QA_CHUNK_TYPES,
    branch_cache,
    discover_qa,
    qa_direct_coverage,
    qa_objective,
    sweep_qa_weights,
)
from rag.retrieval import Retriever

pd.set_option("display.max_colwidth", 140)

bundle = load_corpus()
conn = connect()
with capture_output():
    embedder = Embedder(config())
runtime = Retriever(conn, embedder)
assert runtime.snapshot == bundle["snapshot"]
print("Active snapshot:", runtime.snapshot, "profile:", runtime.profile)
print("Corpus records:", len(bundle["records"]))

Active snapshot: e06d50069151e5b0ecae0450 profile: 3a2ce6e6d726a0f053748717
Corpus records: 5356


## 1. Kiểm tra train và nguồn

Chỉ đọc `data/verified/qa/train/LEGAL_RETRIEVAL_TRAIN.jsonl` ở bước chọn cấu hình. Mapping gold được dùng để **chấm điểm**, không đi vào query hay cơ sở tri thức.

In [3]:
qa_train = discover_qa(bundle, split="train")
assert len(qa_train) == 320
assert len({c["case_family_id"] for c in qa_train}) == len(qa_train)
assert all(c["canonical_groups"] for c in qa_train)
train_hash = digest(qa_train)
source_by_id = {r["id"]: r for r in bundle["records"]}
train_rows = []
for case in qa_train:
    primary = [g["any_of"][0] for g in case["canonical_groups"]]
    train_rows.append(
        {
            "case_type": case.get("case_type", ""),
            "section_scope": ",".join(case.get("section_scope", [])),
            "primary_tariff": any(source_by_id[s]["chunk_type"] == "tariff_entry" for s in primary),
            "primary_count": len(primary),
        }
    )
print("Train:", len(qa_train), "unique queries:", len({c["query"].casefold() for c in qa_train}))
display(
    pd.DataFrame(train_rows)
    .groupby(["case_type", "primary_tariff"])
    .size()
    .rename("count")
    .reset_index()
)
print("Train hash:", train_hash)

Train: 320 unique queries: 320


,case_type,primary_tariff,count
0,cross_section,False,1
1,cross_section,True,6
2,gir,False,31
3,legal_rule,False,9
4,note_exclusion,False,2
5,note_rule,False,80
6,tariff_direct,True,90
7,tariff_plus_note,True,8
8,tt85,False,93


Train hash: 72835489c3b4a413b56e8f379323e91371b1c207404284bebd0fa2ab91370358


## 2. Chọn retriever và trọng số trên train

Thử các họ Dense/Sparse/BM25/FTS/Exact và Weighted RRF như notebook 02. Chọn theo tỷ lệ **đủ mọi căn cứ primary trong Top-10**, rồi Recall và MRR. Đây là chọn retrieval policy, không fine-tune BGE-M3.

In [4]:
with capture_output():
    train_cache = branch_cache(
        runtime,
        qa_train,
        intent="knowledge",
        pool=100,
        filters={"chunk_types": QA_CHUNK_TYPES},
        mask_questioned_codes=True,
    )
family_best = []
for family in CONFIGURATIONS:
    family_best.append(sweep_qa_weights(train_cache, family)[0])
base_choice = max(family_best, key=qa_objective)
base_policy = {
    "configuration": base_choice["configuration"],
    "weights": base_choice["weights"],
    "k": base_choice["k"],
    "pool": 100,
    "reranker": "cross_encoder",
    "rerank_pool": 30,
    "rank_weight": 0.75,
    "graph_before_rerank": False,
    "query_split_weight": 0.0,
}
old_policy = {
    "configuration": "C11",
    "weights": {"dense": 0.30, "sparse": 0.05, "bm25": 0.30, "fts": 0.05, "exact": 0.30},
    "k": 5,
    "pool": 100,
    "reranker": "cross_encoder",
    "rerank_pool": 30,
    "rank_weight": 0.75,
    "graph_before_rerank": False,
    "query_split_weight": 0.0,
}
display(
    pd.DataFrame(family_best)[
        [
            "configuration",
            "weights",
            "k",
            "all_groups@5",
            "all_groups@10",
            "group_recall@10",
            "mrr@10",
        ]
    ]
)
print("Train-selected base:", base_policy)
print("Previous snapshot baseline is evaluated as a transfer, not assumed optimal.")

,configuration,weights,k,all_groups@5,all_groups@10,group_recall@10,mrr@10
0,C1,{'dense': 1.0},5,0.778125,0.812500,0.830313,0.683549
1,C2,{'sparse': 1.0},5,0.743750,0.815625,0.824583,0.663931
2,C3,{'bm25': 1.0},5,0.668750,0.737500,0.756875,0.602372
3,C4,{'fts': 1.0},5,0.281250,0.381250,0.407552,0.183183
4,C5,"{'dense': 0.49933171364562295, 'sparse': 0.500668286354377}",100,0.778125,0.862500,0.870417,0.709707
5,C6,"{'dense': 0.6006673387057762, 'bm25': 0.39933266129422385}",20,0.790625,0.865625,0.876667,0.692397
6,C7,"{'dense': 0.45675863649883747, 'sparse': 0.44711229497230237, 'fts': 0.09612906852886034}",20,0.787500,0.862500,0.868854,0.710935
7,C8,"{'dense': 0.2046589142633734, 'sparse': 0.1988902000539271, 'exact': 0.5964508856826997}",20,0.806250,0.871875,0.876667,0.713826
8,C9,"{'dense': 0.4034468653076416, 'bm25': 0.26559735982308164, 'exact': 0.33095577486927674}",20,0.800000,0.878125,0.886042,0.708392
9,C10,"{'dense': 0.2558906504284244, 'sparse': 0.04600983667132735, 'bm25': 0.200103398101846, 'fts': 0.001635827614227981, 'exact': 0.49636028...",20,0.818750,0.881250,0.891771,0.718127


Train-selected base: {'configuration': 'C13', 'weights': {'dense': 0.25, 'sparse': 0.05, 'bm25': 0.2, 'exact': 0.5}, 'k': 20, 'pool': 100, 'reranker': 'cross_encoder', 'rerank_pool': 30, 'rank_weight': 0.75, 'graph_before_rerank': False, 'query_split_weight': 0.0}
Previous snapshot baseline is evaluated as a transfer, not assumed optimal.


## 3. Ablation trên 320 train

Giữ cùng trọng số đã chọn và thay vị trí graph, kích thước pool, tỷ lệ giữ ứng viên trực tiếp, mức pha thứ hạng gốc và truy vấn bỏ mã nghi vấn. Cả heading/Notes mới đều phải là chunks có văn bản nguồn và locator. Latency bên dưới đo trên mô hình đã nạp, cache reranker có thể làm lần sau nhanh hơn; không dùng latency này để chọn chất lượng.

In [5]:
variants = {
    "baseline_old_snapshot": old_policy,
    "direct_30": base_policy,
    "direct_50": {**base_policy, "rerank_pool": 50},
    "graph_30_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 30,
        "direct_fraction": 0.70,
    },
    "graph_50_50": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.50,
    },
    "graph_50_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
    },
    "graph_50_85": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.85,
    },
    "graph_100_70": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 100,
        "direct_fraction": 0.70,
    },
    "graph_50_pure": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "rank_weight": 1.0,
    },
    "graph_50_neutral_25": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "query_split_weight": 0.25,
    },
    "graph_50_neutral_50": {
        **base_policy,
        "graph_before_rerank": True,
        "rerank_pool": 50,
        "direct_fraction": 0.70,
        "query_split_weight": 0.50,
    },
}

with capture_output():
    runtime.qa_search(qa_train[0]["query"], policy_override=base_policy, include_package=False)
train_experiments = []
for name, policy in variants.items():
    started = time.perf_counter()
    for case in qa_train:
        with capture_output():
            result = runtime.qa_search(case["query"], policy_override=policy, include_package=False)
        metric = qa_direct_coverage(case, result["matches"])
        required = {g["any_of"][0] for g in case["canonical_groups"]}
        pool_ids = set(result["pre_rerank_ids"])
        train_experiments.append(
            {
                "variant": name,
                "case": case["id"],
                "case_type": case.get("case_type", ""),
                "section_scope": ",".join(case.get("section_scope", [])),
                "pool_complete": required <= pool_ids,
                "pool_recall": len(required & pool_ids) / len(required),
                "all_groups@5": metric["all_groups@5"],
                "all_groups@10": metric["all_groups@10"],
                "group_recall@10": metric["group_recall@10"],
                "mrr@10": metric["mrr@10"],
                "elapsed_ms": result["elapsed_ms"],
                "query_split_used": result["query_split_used"],
            }
        )
    print(name, "seconds:", round(time.perf_counter() - started, 1), flush=True)
train_frame = pd.DataFrame(train_experiments)
train_summary = (
    train_frame.groupby("variant")
    .agg(
        count=("case", "count"),
        pool_complete=("pool_complete", "mean"),
        pool_recall=("pool_recall", "mean"),
        **{
            "all_groups@5": ("all_groups@5", "mean"),
            "all_groups@10": ("all_groups@10", "mean"),
            "group_recall@10": ("group_recall@10", "mean"),
            "mrr@10": ("mrr@10", "mean"),
        },
        median_ms=("elapsed_ms", "median"),
        split_queries=("query_split_used", "sum"),
    )
    .reset_index()
)
display(train_summary.sort_values(["all_groups@10", "group_recall@10", "mrr@10"], ascending=False))
display(
    train_frame.groupby(["variant", "case_type"])["all_groups@10"]
    .agg(["mean", "count"])
    .reset_index()
)


baseline_old_snapshot seconds: 167.1


direct_30 seconds: 64.2


direct_50 seconds: 130.7


graph_30_70 seconds: 65.6


graph_50_50 seconds: 69.4


...

graph_50_pure seconds: 30.5


graph_50_neutral_25 seconds: 31.8


graph_50_neutral_50 seconds: 31.4


,variant,count,pool_complete,pool_recall,all_groups@5,all_groups@10,group_recall@10,mrr@10,median_ms,split_queries
6,graph_50_70,320,0.928125,0.931354,0.837500,0.890625,0.894896,0.771138,74.868792,0
8,graph_50_neutral_25,320,0.928125,0.931354,0.837500,0.890625,0.894896,0.771138,75.037875,5
7,graph_50_85,320,0.928125,0.931354,0.837500,0.890625,0.894896,0.770097,74.595125,0
9,graph_50_neutral_50,320,0.928125,0.931354,0.837500,0.890625,0.894896,0.768013,74.994312,5
5,graph_50_50,320,0.925000,0.929792,0.834375,0.887500,0.893333,0.772757,78.622333,0
2,direct_50,320,0.925000,0.928229,0.834375,0.887500,0.891771,0.768943,443.762979,0
3,graph_100_70,320,0.937500,0.941771,0.837500,0.887500,0.891771,0.768631,855.005416,0
10,graph_50_pure,320,0.928125,0.931354,0.837500,0.887500,0.891771,0.761688,75.007208,0
4,graph_30_70,320,0.915625,0.924583,0.837500,0.884375,0.892292,0.774503,106.785291,0
1,direct_30,320,0.918750,0.926146,0.840625,0.884375,0.890729,0.773982,187.161896,0


,variant,case_type,mean,count
0,baseline_old_snapshot,cross_section,0.571429,7
1,baseline_old_snapshot,gir,0.967742,31
2,baseline_old_snapshot,legal_rule,0.444444,9
3,baseline_old_snapshot,note_exclusion,1.000000,2
4,baseline_old_snapshot,note_rule,0.975000,80
...,...,...,...,...
83,graph_50_pure,note_exclusion,1.000000,2
84,graph_50_pure,note_rule,0.987500,80
85,graph_50_pure,tariff_direct,0.733333,90
86,graph_50_pure,tariff_plus_note,1.000000,8


## 4. Đóng băng QA policy trước khi mở test

Lấy phương án tốt nhất trên train trong phạm vi đã thử. Nếu graph không thắng, nó vẫn được báo trong bảng test để chẩn đoán, nhưng không được âm thầm đưa vào runtime.

In [6]:
candidate_names = list(variants)
new_candidate_names = [name for name in variants if name != "baseline_old_snapshot"]
quality = {name: train_summary.set_index("variant").loc[name] for name in candidate_names}
chosen_name = max(
    candidate_names,
    key=lambda name: (
        quality[name]["all_groups@10"],
        quality[name]["group_recall@10"],
        quality[name]["all_groups@5"],
        quality[name]["mrr@10"],
    ),
)
selected_policy = {
    **variants[chosen_name],
    "selection_status": "QA_320_TRAIN_ONLY",
    "train_sha256": train_hash,
    "train_count": len(qa_train),
    "tested_variants": len(variants),
    "selected_variant": chosen_name,
    "objective": ["all_groups@10", "group_recall@10", "all_groups@5", "mrr@10"],
    "candidate_types": QA_CHUNK_TYPES,
    "limitations": "Best among tested policies on supplied train; no model fine-tuning.",
}
hs_selection = read(ROOT / "config/selected_retrieval.json")
assert hs_selection["snapshot"] == runtime.snapshot and hs_selection["profile"] == runtime.profile
selected_config = {**hs_selection, "qa": selected_policy}
assert {k: v for k, v in selected_config.items() if k != "qa"} == {
    k: v for k, v in hs_selection.items() if k != "qa"
}
write_json(ROOT / "config/selected_retrieval.json", selected_config)
frozen_hash = digest(selected_config)
print("Frozen before test:", chosen_name, selected_policy)
print("Configuration digest:", frozen_hash)

Frozen before test: graph_50_70 {'configuration': 'C13', 'weights': {'dense': 0.25, 'sparse': 0.05, 'bm25': 0.2, 'exact': 0.5}, 'k': 20, 'pool': 100, 'reranker': 'cross_encoder', 'rerank_pool': 50, 'rank_weight': 0.75, 'graph_before_rerank': True, 'query_split_weight': 0.0, 'direct_fraction': 0.7, 'selection_status': 'QA_320_TRAIN_ONLY', 'train_sha256': '72835489c3b4a413b56e8f379323e91371b1c207404284bebd0fa2ab91370358', 'train_count': 320, 'tested_variants': 11, 'selected_variant': 'graph_50_70', 'objective': ['all_groups@10', 'group_recall@10', 'all_groups@5', 'mrr@10'], 'candidate_types': ['gir_rule', 'gir_explanation', 'section_note', 'chapter_note', 'subheading_note', 'tariff_entry', 'legal_clause'], 'limitations': 'Best among tested policies on supplied train; no model fine-tuning.'}
Configuration digest: 721fb01053fd8310b4d1dce7317f898bea0e12054a588ff3cc1edc2c5393b5d1


## 5. Test 150 câu sau khi đóng băng

So sánh cách cũ, chính sách tìm trực tiếp, graph tốt nhất trên train, và policy đã chọn. Báo riêng 50 câu cũ đã từng được dùng để chẩn đoán; không dùng test để đổi trọng số hay chọn lại policy.

In [7]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
qa_test = discover_qa(bundle, split="test")
assert len(qa_test) == 150
assert not ({c["case_family_id"] for c in qa_train} & {c["case_family_id"] for c in qa_test})
train_best_direct = max(
    (n for n in new_candidate_names if not variants[n]["graph_before_rerank"]),
    key=lambda n: (
        quality[n]["all_groups@10"],
        quality[n]["group_recall@10"],
        quality[n]["mrr@10"],
    ),
)
train_best_graph = max(
    (n for n in new_candidate_names if variants[n]["graph_before_rerank"]),
    key=lambda n: (
        quality[n]["all_groups@10"],
        quality[n]["group_recall@10"],
        quality[n]["mrr@10"],
    ),
)
test_variants = dict.fromkeys(
    ["baseline_old_snapshot", train_best_direct, train_best_graph, chosen_name]
)
test_rows = []
for name in test_variants:
    policy = variants[name]
    started = time.perf_counter()
    for case in qa_test:
        with capture_output():
            result = runtime.qa_search(case["query"], policy_override=policy, include_package=True)
        metric = qa_direct_coverage(case, result["matches"])
        required = {g["any_of"][0] for g in case["canonical_groups"]}
        pool_ids = set(result["pre_rerank_ids"])
        package_ids = {r["id"] for r in result["package"]}
        test_rows.append(
            {
                "variant": name,
                "case": case["id"],
                "query": case["query"],
                "source_group": "old_50_diagnostic"
                if case.get("legacy_original_index") is not None
                else "new_100",
                "case_type": case.get("case_type", ""),
                "section_scope": ",".join(case.get("section_scope", [])),
                "pool_complete": required <= pool_ids,
                "pool_recall": len(required & pool_ids) / len(required),
                "all_groups@5": metric["all_groups@5"],
                "all_groups@10": metric["all_groups@10"],
                "group_recall@10": metric["group_recall@10"],
                "mrr@10": metric["mrr@10"],
                "package_complete": required <= package_ids,
                "package_recall": len(required & package_ids) / len(required),
                "missing_primary": sorted(required - {r["id"] for r in result["matches"]}),
                "returned_top10": [r["id"] for r in result["matches"]],
                "elapsed_ms": result["elapsed_ms"],
                "query_split_used": result["query_split_used"],
            }
        )
    print("TEST", name, "seconds:", round(time.perf_counter() - started, 1), flush=True)
test_frame = pd.DataFrame(test_rows)
columns = [
    "pool_complete",
    "pool_recall",
    "all_groups@5",
    "all_groups@10",
    "group_recall@10",
    "mrr@10",
    "package_complete",
    "package_recall",
]
test_summary = (
    test_frame.groupby("variant")[columns]
    .mean()
    .join(test_frame.groupby("variant").elapsed_ms.agg(["median", lambda s: s.quantile(0.95)]))
)
test_summary.columns = columns + ["latency_p50_ms", "latency_p95_ms"]
display(test_summary)
display(test_frame.groupby(["source_group", "variant"])[columns].mean())
display(
    test_frame.groupby(["case_type", "variant"])[
        ["all_groups@10", "group_recall@10", "package_complete"]
    ].mean()
)
print(
    "Frozen policy digest unchanged:",
    digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash,
)

TEST baseline_old_snapshot seconds: 105.9


TEST direct_50 seconds: 91.6


TEST graph_50_70 seconds: 59.1


,pool_complete,pool_recall,all_groups@5,all_groups@10,group_recall@10,mrr@10,package_complete,package_recall,latency_p50_ms,latency_p95_ms
variant,,,,,,,,,,
baseline_old_snapshot,0.873333,0.876667,0.80,0.846667,0.861111,0.738127,0.893333,0.896667,739.650000,1134.751336
direct_50,0.893333,0.900000,0.82,0.853333,0.867778,0.742016,0.886667,0.893333,595.720562,940.973723
graph_50_70,0.906667,0.910000,0.82,0.860000,0.874444,0.743738,0.893333,0.896667,258.505958,836.189602


pool_complete  pool_recall  \
source_group      variant                                             
new_100           baseline_old_snapshot           0.87         0.87   
                  direct_50                       0.88         0.88   
                  graph_50_70                     0.89         0.89   
old_50_diagnostic baseline_old_snapshot           0.88         0.89   
                  direct_50                       0.92         0.94   
                  graph_50_70                     0.94         0.95   

                                         all_groups@5  all_groups@10  \
source_group      variant                                              
new_100           baseline_old_snapshot          0.83           0.86   
                  direct_50                      0.85           0.87   
                  graph_50_70                    0.85           0.87   
old_50_diagnostic baseline_old_snapshot          0.74           0.82   
                  direct_50                      0.76           0.82   
                  graph_50_70                    0.76           0.84   

                                         group_recall@10    mrr@10  \
source_group      variant                                            
new_100           baseline_old_snapshot         0.865000  0.791429   
                  direct_50                     0.875000  0.795095   
                  graph_50_70                   0.875000  0.796429   
old_50_diagnostic baseline_old_snapshot         0.853333  0.631524   
                  direct_50                     0.853333  0.635857   
                  graph_50_70                   0.873333  0.638357   

                                         package_complete  package_recall  
source_group      variant                                                  
new_100           baseline_old_snapshot              0.88           0.885  
                  direct_50                          0.88           0.885  
                  graph_50_70                        0.88           0.885  
old_50_diagnostic baseline_old_snapshot              0.92           0.920  
                  direct_50                          0.90           0.910  
                  graph_50_70                        0.92           0.920

all_groups@10  group_recall@10  \
case_type        variant                                                 
cross_section    baseline_old_snapshot       0.333333         0.500000   
                 direct_50                   0.333333         0.500000   
                 graph_50_70                 0.333333         0.666667   
gir              baseline_old_snapshot       0.937500         0.979167   
                 direct_50                   0.937500         0.979167   
                 graph_50_70                 0.937500         0.979167   
legal_rule       baseline_old_snapshot       1.000000         1.000000   
                 direct_50                   1.000000         1.000000   
                 graph_50_70                 1.000000         1.000000   
note_exclusion   baseline_old_snapshot       1.000000         1.000000   
                 direct_50                   1.000000         1.000000   
                 graph_50_70                 1.000000         1.000000   
note_rule        baseline_old_snapshot       1.000000         1.000000   
                 direct_50                   1.000000         1.000000   
                 graph_50_70                 1.000000         1.000000   
tariff_direct    baseline_old_snapshot       0.770270         0.770270   
                 direct_50                   0.783784         0.783784   
                 graph_50_70                 0.783784         0.783784   
tariff_plus_note baseline_old_snapshot       0.884615         0.923077   
                 direct_50                   0.884615         0.923077   
                 graph_50_70                 0.923077         0.942308   
tt85             baseline_old_snapshot       1.000000         1.000000   
                 direct_50                   1.000000         1.000000   
                 graph_50_70                 1.000000         1.000000   

                                        package_complete  
case_type        variant                                  
cross_section    baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_70                    1.000000  
gir              baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_70                    1.000000  
legal_rule       baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_70                    1.000000  
note_exclusion   baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_70                    1.000000  
note_rule        baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_70                    1.000000  
tariff_direct    baseline_old_snapshot          0.810811  
                 direct_50                      0.810811  
                 graph_50_70                    0.810811  
tariff_plus_note baseline_old_snapshot          0.923077  
                 direct_50                      0.884615  
                 graph_50_70                    0.923077  
tt85             baseline_old_snapshot          1.000000  
                 direct_50                      1.000000  
                 graph_50_70                    1.000000

Frozen policy digest unchanged: True


In [8]:
base_test = test_frame[test_frame.variant == train_best_direct].set_index("case")
selected_test = test_frame[test_frame.variant == chosen_name].set_index("case")
comparison = selected_test[
    ["query", "source_group", "case_type", "section_scope", "missing_primary", "returned_top10"]
].copy()
comparison["direct_all@10"] = base_test["all_groups@10"]
comparison["selected_all@10"] = selected_test["all_groups@10"]
comparison["change"] = comparison["selected_all@10"].astype(int) - comparison[
    "direct_all@10"
].astype(int)
print(
    "Gains:",
    (comparison.change > 0).sum(),
    "losses:",
    (comparison.change < 0).sum(),
    "unchanged:",
    (comparison.change == 0).sum(),
)
display(comparison.loc[comparison.change != 0].head(30))
display(
    test_summary[["pool_complete", "all_groups@10", "package_complete"]]
    .style.format("{:.1%}")
    .bar(axis=0, color="#87a9d8", vmin=0, vmax=1)
    .set_caption("QA 150 test: đủ căn cứ trước rerank / Top-10 / gói căn cứ")
)


Gains: 1 losses: 0 unchanged: 149


,query,source_group,case_type,section_scope,missing_primary,returned_top10,direct_all@10,selected_all@10,change
case,,,,,,,,,
LQ012,"Máy in trình bày riêng dù có thể kết nối với máy tính có được phân loại vào 84.71 không, và heading nào cần xem xét?",old_50_diagnostic,tariff_plus_note,XVI,[],"[tt31:SXVI-N-CH84_NOTE_6D, tt31:SXVI-T-8471, tt31:SXVI-T-84669330, tt31:SXVI-T-85284200, tt31:SXVI-T-85286200, tt31:SXVI-T-8470, tt31:SX...",False,True,1


,pool_complete,all_groups@10,package_complete
variant,,,
baseline_old_snapshot,87.3%,84.7%,89.3%
direct_50,89.3%,85.3%,88.7%
graph_50_70,90.7%,86.0%,89.3%


## 6. Runtime và kiểm tra

Đối chiếu runtime dùng cấu hình đã đóng băng với đường thử nghiệm. Kiểm tra chính sách HS không bị thay đổi khi thêm khóa QA; kết quả HS và QA thuộc cùng snapshot.

In [9]:
assert digest(read(ROOT / "config/selected_retrieval.json")) == frozen_hash
with connect() as fresh_conn:
    fresh = Retriever(fresh_conn, embedder)
    assert fresh.qa_policy["selected_variant"] == chosen_name
    for case in qa_train[:5] + qa_test[:5]:
        with capture_output():
            expected = runtime.qa_search(
                case["query"], policy_override=selected_policy, include_package=False
            )
            actual = fresh.qa_search(case["query"], include_package=False)
        assert [r["id"] for r in expected["matches"]] == [r["id"] for r in actual["matches"]]
print("QA runtime parity: 10/10 exact Top-10.")
assert {k: v for k, v in read(ROOT / "config/selected_retrieval.json").items() if k != "qa"} == {
    k: v for k, v in hs_selection.items() if k != "qa"
}
print("HS configuration unchanged by QA selection.")

QA runtime parity: 10/10 exact Top-10.
HS configuration unchanged by QA selection.


In [3]:
import subprocess

validation_env = {**os.environ, "RAG_RUN_DB_TESTS": "1", "TOKENIZERS_PARALLELISM": "false"}
for command in [
    [str(ROOT / ".venv/bin/ruff"), "check", "rag", "tests", "notebooks"],
    [str(ROOT / ".venv/bin/mypy"), "rag"],
    [str(ROOT / ".venv/bin/bandit"), "-r", "rag", "-ll", "-q"],
    [str(ROOT / ".venv/bin/python"), "-m", "pytest", "tests", "-q"],
    ["git", "diff", "--check"],
]:
    checked = subprocess.run(
        command, cwd=ROOT, env=validation_env, capture_output=True, text=True, check=False
    )
    print(
        " ".join(command),
        "exit",
        checked.returncode,
        checked.stdout[-700:] or checked.stderr[-700:] or "PASS",
    )
    assert checked.returncode == 0
selection_now = read(ROOT / "config/selected_retrieval.json")
assert selection_now["snapshot"] == runtime.snapshot
if "frozen_hash" in globals():
    assert digest(selection_now) == frozen_hash
conn.close()
print("Completed: snapshot", bundle["snapshot"], "QA policy", selection_now["qa"]["selected_variant"])

/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/ruff check rag tests notebooks exit 0 All checks passed!



/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/mypy rag exit 0 Success: no issues found in 6 source files



/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/bandit -r rag -ll -q exit 0 PASS


/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/bin/python -m pytest tests -q exit 0 ........................................................                 [100%]
56 passed in 19.69s

git diff --check exit 0 PASS
Completed: snapshot e06d50069151e5b0ecae0450 QA policy graph_50_70


## QA regression on the new active Knowledge Base snapshot

The HS corpus was refreshed from the verified source. Recheck the already selected QA policy on all 150 supplied QA test questions without changing its weights or the frozen HS policy. These are retrieval metrics, not legal conclusions.

In [3]:
import numpy as np

from rag.evaluation import qa_coverage

policy_hash = digest(read(ROOT / "config/selected_retrieval.json"))
assert runtime.snapshot == read(ROOT / "config/selected_retrieval.json")["snapshot"]
qa_test_current = discover_qa(bundle, split="test")
assert len(qa_test_current) == 150
qa_rows = []
for index, case in enumerate(qa_test_current, 1):
    with capture_output() as progress:
        result = runtime.qa_search(case["query"])
    direct = qa_direct_coverage(case, result["matches"])
    package = qa_coverage(case, result["matches"], result["package"])
    qa_rows.append({"id": case["id"], "category": case["category"],
        "direct_all@5": direct["all_groups@5"],
        "direct_all@10": direct["all_groups@10"],
        "direct_group_recall@10": direct["group_recall@10"],
        "direct_mrr@10": direct["mrr@10"],
        "package_all": package["package_all_groups"],
        "package_group_recall": package["package_group_recall"],
        "missing_package": package["missing_package_groups"],
        "latency_ms": result["elapsed_ms"],
        "top10_ids": direct["returned_top10"]})
    if index % 25 == 0:
        print("QA regression", index, "/", len(qa_test_current), flush=True)
assert digest(read(ROOT / "config/selected_retrieval.json")) == policy_hash
frame = pd.DataFrame(qa_rows)
summary = frame.groupby("category").agg(
    questions=("id", "count"),
    direct_all_at_5=("direct_all@5", "mean"),
    direct_all_at_10=("direct_all@10", "mean"),
    direct_group_recall_at_10=("direct_group_recall@10", "mean"),
    direct_mrr_at_10=("direct_mrr@10", "mean"),
    package_all=("package_all", "mean"),
    latency_p50_ms=("latency_ms", lambda x: float(np.percentile(x, 50))),
    latency_p95_ms=("latency_ms", lambda x: float(np.percentile(x, 95))))
display(summary)
print("Overall direct all@10", frame["direct_all@10"].mean(),
      "package all", frame["package_all"].mean())
print("QA policy hash unchanged", policy_hash)
display(frame)


QA regression 25 / 150


QA regression 50 / 150


QA regression 75 / 150


QA regression 100 / 150


QA regression 125 / 150


QA regression 150 / 150


,questions,direct_all_at_5,direct_all_at_10,direct_group_recall_at_10,direct_mrr_at_10,package_all,latency_p50_ms,latency_p95_ms
category,,,,,,,,
cross_section_exclusion,2,0.000000,0.000000,0.500000,0.562500,1.000000,194.764334,200.609533
gir_rule_retrieval,4,0.750000,0.750000,0.916667,0.800000,1.000000,71.139937,84.977671
heading_or_subheading_legal_retrieval,29,0.827586,0.862069,0.862069,0.640805,0.896552,135.882584,381.806917
heading_plus_legal_note,11,0.636364,0.909091,0.909091,0.455411,0.909091,145.205834,204.555354
legal_evidence,55,0.945455,0.981818,0.990909,0.932900,0.981818,136.706000,235.036788
legal_note_retrieval,2,1.000000,1.000000,1.000000,1.000000,1.000000,154.942708,223.394833
multi_source_legal_retrieval,2,1.000000,1.000000,1.000000,1.000000,1.000000,70.944834,71.829683
tariff_lookup,45,0.733333,0.733333,0.733333,0.629630,0.755556,166.664500,286.699675


Overall direct all@10 0.86 package all 0.8933333333333333
QA policy hash unchanged 721fb01053fd8310b4d1dce7317f898bea0e12054a588ff3cc1edc2c5393b5d1


,id,category,direct_all@5,direct_all@10,direct_group_recall@10,direct_mrr@10,package_all,package_group_recall,missing_package,latency_ms,top10_ids
0,LQ001,heading_or_subheading_legal_retrieval,True,True,1.0,1.00,True,1.0,[],1348.899333,"[tt31:HS4:7101, tt31:XIV_CH71_NOTE_11, tt31:XIV_CH71_NOTE_01, tt31:XIV_CH71_NOTE_09, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7103, tt31:HS4:7..."
1,LQ002,heading_or_subheading_legal_retrieval,True,True,1.0,1.00,True,1.0,[],121.631417,"[tt31:HS4:7102, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7103, tt31:HS4:7101, tt31:HS4:7104, tt31:HS4:7111, tt31:XIV_CH71_NOTE_03, tt31:HS8:71..."
2,LQ003,heading_or_subheading_legal_retrieval,True,True,1.0,0.25,True,1.0,[],111.268791,"[tt31:HS4:7102, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7101, tt31:HS4:7103, tt31:XIV_CH71_NOTE_11, tt31:HS4:7104, tt31:SXVI-N-SXVI_NOTE_1Q, ..."
3,LQ004,heading_or_subheading_legal_retrieval,True,True,1.0,0.25,True,1.0,[],112.433292,"[tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS4:7102, tt31:XIV_CH71_NOTE_11, tt31:HS4:7104, tt31:XIV_CH71_NOTE_03, tt31:HS4:7101, tt31:HS4:7103, tt3..."
4,LQ005,heading_or_subheading_legal_retrieval,True,True,1.0,1.00,True,1.0,[],106.160708,"[tt31:HS4:7108, tt31:HS4:7111, tt31:HS4:7106, tt31:HS4:7110, tt31:HS4:7102, tt31:XIV_CH71_NOTE_10, tt31:SXVI-N-SXVI_NOTE_1F, tt31:HS8:71..."
...,...,...,...,...,...,...,...,...,...,...,...
145,LQ146,legal_evidence,True,True,1.0,1.00,True,1.0,[],69.352083,"[tt85:TT85-A12-3, tt85:TT85-A12-4, tt85:TT85-A11-3.b, tt85:TT85-A11-3, tt85:TT85-A12-6, tt85:TT85-A12-1, tt85:TT85-A12-2, tt85:TT85-A6-5..."
146,LQ147,legal_evidence,True,True,1.0,1.00,True,1.0,[],68.121708,"[tt85:TT85-A12-6, tt85:TT85-A5-1, tt85:TT85-A11-3, tt85:TT85-A11-3.b, tt85:TT85-A12-3, tt85:TT85-A12-1, tt85:TT85-A12-2, tt85:TT85-A12-4..."
147,LQ148,legal_evidence,True,True,1.0,1.00,True,1.0,[],72.999125,"[tt85:TT85-A12-7, tt85:TT85-A11-3.b, tt85:TT85-A11-3, tt85:TT85-A12-3, tt85:TT85-A17-2, tt85:TT85-A12-6, tt85:TT85-A3-1, tt85:TT85-A11-3..."
148,LQ149,legal_evidence,True,True,1.0,1.00,True,1.0,[],128.364708,"[tt85:TT85-A15-1, tt85:TT85-A10-2, tt85:TT85-A1-1, tt85:TT85-A12-3, tt85:TT85-A12-5, tt85:TT85-A15-2, tt85:TT85-A1-2, tt85:TT85-A16-2, t..."
